In [4]:
import duckdb
from pathlib import Path
from ravelry_scraper.constants import DATA_DIR

In [11]:
con = duckdb.connect()

# Register Parquet datasets as SQL views
for file in DATA_DIR.glob("**/*.parquet"):
    name = f"{file.parent.name}_{file.stem}"
    con.execute(f"""
        CREATE OR REPLACE VIEW {name} AS
        SELECT *
        FROM read_parquet('{file}')
    """)

In [12]:
con.sql("SELECT COUNT(*) AS total_patterns FROM raw_patterns").df()

,total_patterns
0,1261768


In [14]:
con.execute(f"""
    SELECT
        COUNT(*) AS total_patterns,
        COUNT(DISTINCT id) AS unique_patterns,
        COUNT(price) AS has_price,
        COUNT(rating_average) AS has_rating,
        COUNT(difficulty_average) AS has_difficulty,
        COUNT(projects_count) AS has_projects,
        COUNT(favorites_count) AS has_favorites,
        COUNT(yardage) AS has_yardage
    FROM raw_patterns
""").fetchdf()

,total_patterns,unique_patterns,has_price,has_rating,has_difficulty,has_projects,has_favorites,has_yardage
0,1261768,1261768,614427,1261768,1261768,1261768,1261768,804551


In [16]:
con.execute(f"""
    SELECT
        free,
        COUNT(*) AS total_patterns,
        COUNT(price) AS has_price,
        AVG(price) AS avg_price,
        AVG(rating_average) AS avg_rating,
        AVG(projects_count) AS avg_projects
    FROM raw_patterns
    GROUP BY free
    ORDER BY free
""").fetchdf()

,free,total_patterns,has_price,avg_price,avg_rating,avg_projects
0,False,939919,582216,15.032290,1.021012,15.126277
1,True,321833,32210,4.348432,1.475944,33.377892
2,<NA>,16,1,10.000000,2.263884,21.562500
